In [1]:
from typing import Optional, Union, Sequence, Literal
import torch
import torch.nn as nn
from typing_extensions import TypeAlias

In [2]:
from torch_pointcloud.layers.activations import ActLike, get_act
from torch_pointcloud.layers.dropouts import get_dropout
from torch_pointcloud.layers.norms import NormLike, get_norm

In [3]:
# ActLike: TypeAlias = Union[str, nn.Module]
# NormLike: TypeAlias = Union[str, nn.Module]

In [4]:
def _validate_block_order(
    order: str,
    layer_id: str,
    has_act: bool,
    has_norm: bool,
    has_dropout: bool
) -> None:
    """Validates the layer order string.
    
    Args:
        order: Order string (e.g., 'land' for linear block)
        layer_id: Main layer identifier ('l' for linear, 'c' for conv)
        has_act: Whether activation is present
        has_norm: Whether normalization is present
        has_dropout: Whether dropout is present
    """
    valid_chars = {layer_id, 'a', 'n', 'd'}
    
    if not all(o in valid_chars for o in order):
        raise ValueError(f"Invalid characters in order string. Valid characters are: {valid_chars}")
    
    if len(order) != len(set(order)):
        raise ValueError("The 'order' sequence must not contain duplicate elements.")
    
    if layer_id not in order:
        raise ValueError(f"The main layer '{layer_id}' must be present in the order sequence.")
    
    if has_act and 'a' not in order:
        raise ValueError("Activation layer 'a' must be in order when activation is specified.")
    
    if has_norm and 'n' not in order:
        raise ValueError("Normalization layer 'n' must be in order when normalization is specified.")
    
    if has_dropout and 'd' not in order:
        raise ValueError("Dropout layer 'd' must be in order when dropout is specified.")

def linear_block(
    in_features: int,
    out_features: int,
    bias: bool = True,
    act: Optional[ActLike] = "relu",
    norm: Optional[NormLike] = "batch_norm1d",
    dropout: Optional[float] = 0.0,
    order: Union[str, Sequence[Literal["a", "l", "n", "d"]]] = "land",
) -> nn.Sequential:
    """Creates a Sequential linear block with customizable layer order.
    
    Args:
        in_features: Number of input features
        out_features: Number of output features
        bias: Whether to include bias in linear layer
        act: Activation function specification
        norm: Normalization layer specification
        dropout: Dropout probability
        order: Order of layers (l=linear, a=activation, n=norm, d=dropout)
    
    Returns:
        nn.Sequential: Sequential container of layers in specified order
    """
    order = order if isinstance(order, str) else "".join(order)
    
    has_act = act is not None
    has_norm = norm is not None
    has_dropout = dropout is not None
    
    _validate_block_order(order, 'l', has_act, has_norm, has_dropout)
    
    # Create layer instances
    layer_instances = {
        'l': nn.Linear(in_features, out_features, bias=bias),
        'a': get_act(act) if has_act else None,
        'n': get_norm(norm, out_features) if has_norm else None,
        'd': get_dropout("dropout", dropout) if has_dropout else None
    }
    
    # Create sequential container with layers in specified order
    layers_ordered = [
        layer_instances[layer_id]
        for layer_id in order
        if layer_instances[layer_id] is not None
    ]
    
    return nn.Sequential(*layers_ordered)

def conv1d_block(
    in_channels: int,
    out_channels: int,
    kernel_size: int,
    stride: int = 1,
    padding: Union[str, int] = 0,
    dilation: int = 1,
    groups: int = 1,
    bias: bool = True,
    act: Optional[ActLike] = "relu",
    norm: Optional[NormLike] = "batch_norm1d",
    dropout: Optional[float] = 0.0,
    order: Union[str, Sequence[Literal["a", "c", "n", "d"]]] = "cand",
) -> nn.Sequential:
    """Creates a Sequential 1D convolution block with customizable layer order.
    
    Args:
        in_channels: Number of input channels
        out_channels: Number of output channels
        kernel_size: Size of the convolving kernel
        stride: Stride of the convolution
        padding: Padding added to both sides of the input
        dilation: Spacing between kernel elements
        groups: Number of blocked connections from input to output channels
        bias: Whether to include bias
        act: Activation function specification
        norm: Normalization layer specification
        dropout: Dropout probability
        order: Order of layers (c=conv, a=activation, n=norm, d=dropout)
    
    Returns:
        nn.Sequential: Sequential container of layers in specified order
    """
    order = order if isinstance(order, str) else "".join(order)
    
    has_act = act is not None
    has_norm = norm is not None
    has_dropout = dropout is not None
    
    _validate_block_order(order, 'c', has_act, has_norm, has_dropout)
    
    # Create layer instances
    layer_instances = {
        'c': nn.Conv1d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            dilation=dilation,
            groups=groups,
            bias=bias
        ),
        'a': get_act(act) if has_act else None,
        'n': get_norm(norm, out_channels) if has_norm else None,
        'd': get_dropout("dropout", dropout) if has_dropout else None
    }
    
    # Create sequential container with layers in specified order
    layers_ordered = [
        layer_instances[layer_id]
        for layer_id in order
        if layer_instances[layer_id] is not None
    ]
    
    return nn.Sequential(*layers_ordered)

In [5]:
block = linear_block(
    in_features=10,
    out_features=20,
    # dropout=None,
    order="aldn"
)

block

Sequential(
  (0): ReLU()
  (1): Linear(in_features=10, out_features=20, bias=True)
  (2): Dropout(p=0.0, inplace=False)
  (3): BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)

In [6]:
block = conv1d_block(
    in_channels=3,
    out_channels=6,
    kernel_size=3,
    stride=1,
    padding=1,
    dilation=1,
    groups=1,
    bias=True,
    act="relu",
    norm="batch_norm1d",
    dropout=0.1,
    order="dcan"
)

block

Sequential(
  (0): Dropout(p=0.1, inplace=False)
  (1): Conv1d(3, 6, kernel_size=(3,), stride=(1,), padding=(1,))
  (2): ReLU()
  (3): BatchNorm1d(6, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)

In [7]:
def _validate_block_order(
    order: str,
    layers: dict,
) -> None:
    """Validates the layer order string against provided layers.
    
    Args:
        order: Order string (e.g., 'land' for linear block)
        layer_id: Main layer identifier ('l' for linear, 'c' for conv)
        layers: Dictionary of layer instances
    """
    if not all(o in layers for o in order):
        valid_layer_ids = ", ".join([f"{k!r}" for k in layers.keys()])
        raise ValueError(f"Invalid order sequence. Got order {order!r}, but valid layer IDs are {valid_layer_ids}.")
    
    if len(order) != len(set(order)):
        raise ValueError("The order sequence must not contain duplicate elements.")
    
    for layer_id, layer in layers.items():
        if layer is not None and layer_id not in order:
            raise ValueError(f"Layer {layer_id!r} must be in the order sequence. Got order {order!r}.")

In [8]:
_validate_block_order(
    order="landx",
    layers={'l': 1, 'a': 2, 'n': 3, 'd': None, 'x': 4}
)

In [9]:
def linear_block(
    in_features: int,
    out_features: int,
    bias: bool = True,
    act: Optional[ActLike] = "relu",
    norm: Optional[NormLike] = "batch_norm1d",
    dropout: Optional[float] = 0.0,
    order: Union[str, Sequence[Literal["a", "l", "n", "d"]]] = "land",
) -> nn.Sequential:
    """Creates a Sequential linear block with customizable layer order.
    
    Args:
        in_features: Number of input features
        out_features: Number of output features
        bias: Whether to include bias in linear layer
        act: Activation function specification
        norm: Normalization layer specification
        dropout: Dropout probability
        order: Order of layers (l=linear, a=activation, n=norm, d=dropout)
    
    Returns:
        nn.Sequential: Sequential container of layers in specified order
    """
    order = order if isinstance(order, str) else "".join(order)
    
    # Create layer instances with proper typing
    layers = {
        'l': nn.Linear(in_features, out_features, bias=bias),
        'a': get_act(act) if act is not None else None,
        'n': get_norm(norm, out_features) if norm is not None else None,
        'd': get_dropout(dropout) if dropout is not None else None
    }
    
    _validate_block_order(order, layers)
    
    # Create sequential container with layers in specified order
    layers_ordered = [
        layers[layer_id]
        for layer_id in order
        if layers[layer_id] is not None
    ]
    
    return nn.Sequential(*layers_ordered)

In [ ]:
import torch.nn.functional as F
import functools    



block = linear_block(
    in_features=10,
    out_features=20,
    dropout=nn.AlphaDropout(0.1),
    act=functools.partial(nn.LeakyReLU, negative_slope=0.2),
    norm=functools.partial(nn.BatchNorm1d, momentum=0.1),
    order="land"
)

block

Sequential(
  (0): Linear(in_features=10, out_features=20, bias=True)
  (1): LeakyReLU(negative_slope=0.2)
  (2): BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): AlphaDropout(p=0.1, inplace=False)
)